In [8]:
import re
import pandas as pd
from pathlib import Path

RAW_DIR = Path("../data/raw")
OUTPUT   = Path("../data/processed/lectures.csv")
OUTPUT_KSS = Path("../data/processed/lectures_kss.csv")

## 1. 원본 파싱 → lectures.csv

In [ ]:
LINE_RE = re.compile(r"^<(\d{2}:\d{2}:\d{2})>\s+(\S+):\s*(.*)$")

rows = []
for txt in sorted(RAW_DIR.glob("*.txt")):
    date, *rest = txt.stem.split("_", 1)
    lecture_id = rest[0] if rest else txt.stem

    for line in txt.read_text(encoding="utf-8").splitlines():
        m = LINE_RE.match(line.strip())
        if not m:
            continue
        rows.append({
            "lecture_id" : lecture_id,
            "date"       : date,
            "timestamp"  : m.group(1),
            "speaker_id" : m.group(2),
            "text_raw"   : m.group(3),
        })

df = pd.DataFrame(rows, columns=["lecture_id", "date", "timestamp", "speaker_id", "text_raw"])
print(f"{len(df):,} rows, {df['lecture_id'].nunique()} lectures")
df.head()

In [ ]:
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUTPUT, index=False, encoding="utf-8")
print(f"Saved → {OUTPUT.resolve()}")

## 2. KSS 문장 분리 → lectures_kss.csv

파일 단위로 전체 발화를 이어붙인 뒤 KSS로 문장을 분리한다.  
각 문장의 `timestamp` / `speaker_id` 는 해당 문장이 **시작되는 원본 라인**의 값을 사용한다.

In [9]:
import kss

def split_with_timestamps(group: pd.DataFrame) -> list[dict]:
    """파일 하나(group)를 KSS 로 문장 분리하고 원본 타임스탬프를 매핑한다."""
    texts      = group["text_raw"].tolist()
    timestamps = group["timestamp"].tolist()
    speakers   = group["speaker_id"].tolist()

    # 원본 라인을 공백으로 이어붙이고 각 라인의 시작 오프셋을 기록
    joined  = ""
    offsets = []          # offsets[i] = 라인 i 가 joined 에서 시작하는 위치
    for t in texts:
        offsets.append(len(joined))
        joined += t + " "
    joined = joined.rstrip()

    sentences = kss.split_sentences(joined, backend="auto")

    results  = []
    search_from = 0

    for sent in sentences:
        sent_start = joined.find(sent, search_from)
        if sent_start == -1:
            sent_start = search_from

        # 이진 탐색으로 해당 위치를 포함하는 원본 라인 인덱스 산출
        lo, hi = 0, len(offsets) - 1
        while lo < hi:
            mid = (lo + hi + 1) // 2
            if offsets[mid] <= sent_start:
                lo = mid
            else:
                hi = mid - 1
        line_idx = lo

        results.append({
            "lecture_id" : group["lecture_id"].iloc[0],
            "date"       : group["date"].iloc[0],
            "timestamp"  : timestamps[line_idx],
            "speaker_id" : speakers[line_idx],
            "text_raw"   : sent,
        })
        search_from = sent_start + len(sent)

    return results

In [10]:
kss_rows = []
groups = df.groupby(["lecture_id", "date"], sort=False)
for (lecture_id, date), group in groups:
    kss_rows.extend(split_with_timestamps(group.reset_index(drop=True)))
    print(f"{lecture_id} {date}: {len(group)} lines → {len(kss_rows)} sentences so far")

df_kss = pd.DataFrame(kss_rows, columns=["lecture_id", "date", "timestamp", "speaker_id", "text_raw"])
print(f"\nTotal: {len(df_kss):,} sentences")
df_kss.head(10)

[Kss]: Oh! You have mecab in your environment. Kss will take this as a backend! :D



kdt-backendj-21th 2026-02-02: 1484 lines → 1613 sentences so far
kdt-backendj-21th 2026-02-03: 1860 lines → 3671 sentences so far
kdt-backendj-21th 2026-02-04: 1771 lines → 5658 sentences so far
kdt-backendj-21th 2026-02-05: 1656 lines → 7474 sentences so far
kdt-backendj-21th 2026-02-06: 1671 lines → 9338 sentences so far
kdt-backendj-21th 2026-02-09: 1555 lines → 10984 sentences so far
kdt-backendj-21th 2026-02-10: 1672 lines → 12799 sentences so far
kdt-backendj-21th 2026-02-11: 1560 lines → 14459 sentences so far
kdt-backendj-21th 2026-02-12: 986 lines → 15496 sentences so far
kdt-backendj-21th 2026-02-13: 1543 lines → 17039 sentences so far
kdt-backendj-21th 2026-02-23: 1469 lines → 18564 sentences so far
kdt-backendj-21th 2026-02-24: 1578 lines → 20240 sentences so far
kdt-backendj-21th 2026-02-25: 1298 lines → 21570 sentences so far
kdt-backendj-21th 2026-02-26: 1017 lines → 22584 sentences so far
kdt-backendj-21th 2026-02-27: 1636 lines → 24238 sentences so far

Total: 24,238 s

,lecture_id,date,timestamp,speaker_id,text_raw
0,kdt-backendj-21th,2026-02-02,09:11:17,b54f46b0,여러분 오늘 수업 진행하도록 하겠습니다.
1,kdt-backendj-21th,2026-02-02,09:11:17,b54f46b0,저희가 이제 오늘 1차 잡바 마지막 날입니다.
2,kdt-backendj-21th,2026-02-02,09:11:17,b54f46b0,그래서 지난 시간에 제너릭 타입을 이용을 해서 커스텀 컬렉션을 활용한 크루드 방법을...
3,kdt-backendj-21th,2026-02-02,09:11:17,b54f46b0,"자바는 NIO 패키지가 있고 그 다음에 NIO2라는 패키지를 가지고 있는데, 이 N..."
4,kdt-backendj-21th,2026-02-02,09:11:18,b54f46b0,그래서 이걸 세 가지 섹션으로 나눠서 진행을 하고 있습니다.
5,kdt-backendj-21th,2026-02-02,09:11:18,b54f46b0,오늘 이런 것들을 조금 살펴보고 오후에는 효울씨 입실하기 전에 데이터베이스 설치하고...
6,kdt-backendj-21th,2026-02-02,09:11:33,b54f46b0,일단은 저희가 오늘 이제 수업할 내용은 자바 아이오 패키지가 이렇게 구현이 되어 있...
7,kdt-backendj-21th,2026-02-02,09:11:57,b54f46b0,"지금 다양한 섹션으로 구현할 수가 있는데, 여기 보시면 잡아이오가 있고요."
8,kdt-backendj-21th,2026-02-02,09:12:04,b54f46b0,아이오 패키지가 있고 그다음에 밑에 보시면은 자바 NIO라는 패키지가 있죠.
9,kdt-backendj-21th,2026-02-02,09:12:04,b54f46b0,그 다음에 스트림 단위로 구현을 해주는 자바 NIO 여기 보면 파일이라고 하는 객체...


In [11]:
df_kss.to_csv(OUTPUT_KSS, index=False, encoding="utf-8")
print(f"Saved → {OUTPUT_KSS.resolve()}")

Saved → /Users/parkdahyeon/Documents/my_ws/Lecturesight-Lab/data/processed/lectures_kss.csv
